# Bronze Data Seed - Italy (lh_poc_ita_bronze)

This notebook generates synthetic Bronze layer data for the Italy POC.
Run this in the **EMEA_GDP_POC_ITADBT** workspace with `lh_poc_ita_bronze` as the default Lakehouse.

Tables created (6 total):
- **Sales domain**: client, branch, sales_order
- **Finance domain**: account, cost_center, journal_entry

In [ ]:
from decimal import Decimal
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, concat, expr, when, rand, floor, date_sub, current_date,
    monotonically_increasing_id, row_number
)
from pyspark.sql.window import Window
from pyspark.sql.types import *
from datetime import datetime, timedelta
import random

## Sales Domain

In [ ]:
# 1. CLIENT - Master data (~5000 rows)
num_clients = 5000

client_types = ['ENTERPRISE', 'SMB', 'INDIVIDUAL', 'GOVERNMENT']
country_codes = ['ITA', 'ITA', 'ITA', 'DEU', 'FRA']  # 60% ITA

client_data = []
for i in range(1, num_clients + 1):
    client_data.append((
        i,
        f'Client_ITA_{i}',
        client_types[i % 4],
        country_codes[i % 5],
        datetime(2026, 1, 1) - timedelta(days=i % 1000)
    ))

client_schema = StructType([
    StructField('client_id', IntegerType(), False),
    StructField('client_name', StringType(), False),
    StructField('client_type', StringType(), False),
    StructField('country_code', StringType(), False),
    StructField('created_at', TimestampType(), False)
])

df_client = spark.createDataFrame(client_data, schema=client_schema)
df_client.write.mode('overwrite').format('delta').saveAsTable('lh_poc_ita_bronze.client')
print(f'client: {df_client.count()} rows written')

In [ ]:
# 2. BRANCH - Master data (~200 rows)
num_branches = 200

cities = ['Milan', 'Rome', 'Turin', 'Naples', 'Florence', 'Bologna', 'Genoa', 'Palermo']
regions = ['Lombardy', 'Lazio', 'Piedmont', 'Campania', 'Tuscany', 'Emilia-Romagna', 'Liguria', 'Sicily']

branch_data = []
for i in range(1, num_branches + 1):
    branch_data.append((
        i,
        f'Branch_ITA_{i}',
        cities[i % 8],
        regions[i % 8],
        i % 10 != 0  # 90% active
    ))

branch_schema = StructType([
    StructField('branch_id', IntegerType(), False),
    StructField('branch_name', StringType(), False),
    StructField('branch_city', StringType(), False),
    StructField('branch_region', StringType(), False),
    StructField('is_active', BooleanType(), False)
])

df_branch = spark.createDataFrame(branch_data, schema=branch_schema)
df_branch.write.mode('overwrite').format('delta').saveAsTable('lh_poc_ita_bronze.branch')
print(f'branch: {df_branch.count()} rows written')

In [ ]:
# 3. SALES_ORDER - Transactional (~10000 rows)
# FIX: amount uses Decimal() instead of float to satisfy DecimalType(18,2) schema
num_orders = 10000

currencies = ['EUR', 'EUR', 'CHF']  # mostly EUR
statuses = ['COMPLETED', 'COMPLETED', 'PENDING', 'SHIPPED', 'CANCELLED']

order_data = []
for i in range(1, num_orders + 1):
    order_data.append((
        i,
        (i % num_clients) + 1,
        (i % num_branches) + 1,
        (datetime(2026, 5, 1) - timedelta(days=i % 365)).date(),
        Decimal(str(round(50.0 + (i % 9950), 2))),
        currencies[i % 3],
        statuses[i % 5]
    ))

order_schema = StructType([
    StructField('order_id', IntegerType(), False),
    StructField('client_id', IntegerType(), False),
    StructField('branch_id', IntegerType(), False),
    StructField('order_date', DateType(), False),
    StructField('amount', DecimalType(18, 2), False),
    StructField('currency_code', StringType(), False),
    StructField('order_status', StringType(), False)
])

df_order = spark.createDataFrame(order_data, schema=order_schema)
df_order.write.mode('overwrite').format('delta').saveAsTable('lh_poc_ita_bronze.sales_order')
print(f'sales_order: {df_order.count()} rows written')

## Finance Domain

In [ ]:
# 4. ACCOUNT - Master data (~500 rows)
num_accounts = 500

account_types = ['ASSET', 'LIABILITY', 'EQUITY', 'REVENUE', 'EXPENSE']
account_categories = ['CURRENT_ASSET', 'FIXED_ASSET', 'SHORT_TERM_LIABILITY',
                      'LONG_TERM_LIABILITY', 'OPERATING_REVENUE', 'OPERATING_EXPENSE']

account_data = []
for i in range(1, num_accounts + 1):
    account_data.append((
        i,
        f'Account_ITA_{i}',
        account_types[i % 5],
        account_categories[i % 6],
        i % 20 != 0  # 95% active
    ))

account_schema = StructType([
    StructField('account_id', IntegerType(), False),
    StructField('account_name', StringType(), False),
    StructField('account_type', StringType(), False),
    StructField('account_category', StringType(), False),
    StructField('is_active', BooleanType(), False)
])

df_account = spark.createDataFrame(account_data, schema=account_schema)
df_account.write.mode('overwrite').format('delta').saveAsTable('lh_poc_ita_bronze.account')
print(f'account: {df_account.count()} rows written')

In [ ]:
# 5. COST_CENTER - Master data (~100 rows)
num_cost_centers = 100

departments = ['Sales', 'Marketing', 'Finance', 'HR', 'IT', 'Operations', 'Legal', 'R&D']

cc_data = []
for i in range(1, num_cost_centers + 1):
    cc_data.append((
        i,
        f'CC_ITA_{i}',
        departments[i % 8],
        f'Manager_{i}',
        i % 15 != 0  # ~93% active
    ))

cc_schema = StructType([
    StructField('cost_center_id', IntegerType(), False),
    StructField('cost_center_name', StringType(), False),
    StructField('department', StringType(), False),
    StructField('manager_name', StringType(), False),
    StructField('is_active', BooleanType(), False)
])

df_cc = spark.createDataFrame(cc_data, schema=cc_schema)
df_cc.write.mode('overwrite').format('delta').saveAsTable('lh_poc_ita_bronze.cost_center')
print(f'cost_center: {df_cc.count()} rows written')

In [ ]:
# 6. JOURNAL_ENTRY - Transactional (~10000 rows)
# FIX: debit_amount / credit_amount use Decimal() instead of float to satisfy DecimalType(18,2) schema
num_entries = 10000

descriptions = [
    'Salary payment', 'Office supplies', 'Client invoice',
    'Tax provision', 'Depreciation', 'Consulting fees'
]

entry_data = []
for i in range(1, num_entries + 1):
    debit = Decimal(str(round(100.0 + (i % 9900), 2))) if i % 2 == 0 else Decimal('0.00')
    credit = Decimal(str(round(100.0 + (i % 9900), 2))) if i % 2 == 1 else Decimal('0.00')
    entry_data.append((
        i,
        (i % num_accounts) + 1,
        (i % num_cost_centers) + 1,
        (datetime(2026, 5, 1) - timedelta(days=i % 365)).date(),
        debit,
        credit,
        'EUR' if i % 2 == 0 else 'CHF',
        f'Journal entry {i} - {descriptions[i % 6]}'
    ))

entry_schema = StructType([
    StructField('entry_id', IntegerType(), False),
    StructField('account_id', IntegerType(), False),
    StructField('cost_center_id', IntegerType(), False),
    StructField('entry_date', DateType(), False),
    StructField('debit_amount', DecimalType(18, 2), False),
    StructField('credit_amount', DecimalType(18, 2), False),
    StructField('currency_code', StringType(), False),
    StructField('description', StringType(), False)
])

df_entry = spark.createDataFrame(entry_data, schema=entry_schema)
df_entry.write.mode('overwrite').format('delta').saveAsTable('lh_poc_ita_bronze.journal_entry')
print(f'journal_entry: {df_entry.count()} rows written')

In [ ]:
# Verification - show row counts
tables = ['client', 'branch', 'sales_order', 'account', 'cost_center', 'journal_entry']
print('\n=== Bronze Layer Row Counts ===')
for t in tables:
    count = spark.table(f'lh_poc_ita_bronze.{t}').count()
    print(f'  {t}: {count:,} rows')
print('=== Done ===')